# Employee Attrition Prediction Model
**Author:** Oluwatobi (Tobi) Abanishe  
**Dataset:** IBM HR Analytics Employee Attrition Dataset  
**Source:** https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset

---

## Business Question
> *Which employees are at risk of leaving, and what can the business actually do about it?*

This notebook builds a logistic regression classifier to predict voluntary employee attrition, identifies the top drivers of departure, and translates the findings into concrete business recommendations.

---
## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# Chart styling
plt.rcParams.update({
    'figure.facecolor': '#0D1117',
    'axes.facecolor': '#161B22',
    'axes.edgecolor': '#30363D',
    'axes.labelcolor': '#8B949E',
    'xtick.color': '#8B949E',
    'ytick.color': '#8B949E',
    'text.color': '#E6EDF3',
    'grid.color': '#21262D',
    'grid.linewidth': 0.5,
    'font.family': 'sans-serif',
    'font.size': 11
})
AMBER = '#F0883E'
TEAL  = '#39C5AA'
BLUE  = '#58A6FF'
RED   = '#F85149'
MUTED = '#8B949E'

print('Libraries loaded.')

---
## 2. Load and Inspect the Data

In [ ]:
df = pd.read_csv('../data/WA_Fn-UseC_-HR-Employee-Attrition.csv')

print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')
print(f'\nAttrition breakdown:')
print(df['Attrition'].value_counts())
print(f'\nAttrition rate: {df["Attrition"].value_counts(normalize=True)["Yes"]:.1%}')

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'No missing values found.')

In [ ]:
df.head()

---
## 3. Exploratory Data Analysis

In [ ]:
# Overall attrition rate
attrition_rate = df['Attrition'].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Stayed', 'Left'], attrition_rate.values,
              color=[TEAL, AMBER], width=0.5, edgecolor='none')
ax.set_title('Overall Attrition Rate', color='#E6EDF3', fontsize=13, pad=12)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylim(0, 100)
for bar, val in zip(bars, attrition_rate.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f'{val:.1f}%', ha='center', color='#E6EDF3', fontsize=11)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Attrition by overtime
overtime_attr = df.groupby('OverTime')['Attrition'].apply(
    lambda x: (x == 'Yes').mean() * 100
).reset_index()
overtime_attr.columns = ['OverTime', 'AttritionRate']

fig, ax = plt.subplots(figsize=(6, 4))
colors = [TEAL, AMBER]
bars = ax.bar(overtime_attr['OverTime'], overtime_attr['AttritionRate'],
              color=colors, width=0.4, edgecolor='none')
ax.set_title('Attrition Rate by Overtime Status', color='#E6EDF3', fontsize=13, pad=12)
ax.set_xlabel('Overtime')
ax.set_ylabel('Attrition Rate (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars, overtime_attr['AttritionRate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', color='#E6EDF3', fontsize=11)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

no_ot  = overtime_attr[overtime_attr['OverTime']=='No']['AttritionRate'].values[0]
yes_ot = overtime_attr[overtime_attr['OverTime']=='Yes']['AttritionRate'].values[0]
print(f'Employees on overtime are {yes_ot/no_ot:.1f}x more likely to leave.')

In [ ]:
# Attrition by job role
role_attr = df.groupby('JobRole')['Attrition'].apply(
    lambda x: (x == 'Yes').mean() * 100
).sort_values(ascending=True).reset_index()
role_attr.columns = ['JobRole', 'AttritionRate']

fig, ax = plt.subplots(figsize=(8, 5))
bar_colors = [AMBER if v > 20 else TEAL for v in role_attr['AttritionRate']]
ax.barh(role_attr['JobRole'], role_attr['AttritionRate'],
        color=bar_colors, edgecolor='none')
ax.set_title('Attrition Rate by Job Role', color='#E6EDF3', fontsize=13, pad=12)
ax.set_xlabel('Attrition Rate (%)')
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.axvline(x=16.1, color=MUTED, linestyle='--', linewidth=0.8, label='Overall avg (16.1%)')
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Attrition by years in current role
fig, ax = plt.subplots(figsize=(10, 4))
role_tenure = df.groupby('YearsInCurrentRole')['Attrition'].apply(
    lambda x: (x == 'Yes').mean() * 100
).reset_index()
role_tenure.columns = ['YearsInCurrentRole', 'AttritionRate']

ax.bar(role_tenure['YearsInCurrentRole'], role_tenure['AttritionRate'],
       color=AMBER, width=0.6, edgecolor='none', alpha=0.85)
ax.set_title('Attrition Rate by Years in Current Role', color='#E6EDF3', fontsize=13, pad=12)
ax.set_xlabel('Years in Current Role')
ax.set_ylabel('Attrition Rate (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Monthly income distribution by attrition
fig, ax = plt.subplots(figsize=(8, 4))
for label, color in [('No', TEAL), ('Yes', AMBER)]:
    subset = df[df['Attrition'] == label]['MonthlyIncome']
    ax.hist(subset, bins=30, color=color, alpha=0.6,
            label=f'Attrition = {label}', edgecolor='none')
ax.axvline(x=3500, color=RED, linestyle='--', linewidth=1, label='$3,500 threshold')
ax.set_title('Monthly Income Distribution by Attrition', color='#E6EDF3', fontsize=13, pad=12)
ax.set_xlabel('Monthly Income ($)')
ax.set_ylabel('Employee Count')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

---
## 4. Data Preprocessing

In [ ]:
# Drop columns that are constant or non-informative
cols_to_drop = ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
df_clean = df.drop(columns=cols_to_drop)

# Encode target variable
df_clean['Attrition'] = (df_clean['Attrition'] == 'Yes').astype(int)

# Encode binary categorical columns
df_clean['OverTime'] = (df_clean['OverTime'] == 'Yes').astype(int)

# One-hot encode remaining categorical columns
cat_cols = df_clean.select_dtypes(include='object').columns.tolist()
print(f'Columns to one-hot encode: {cat_cols}')
df_encoded = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)

print(f'\nFinal feature set: {df_encoded.shape[1] - 1} features')
print(f'Final dataset shape: {df_encoded.shape}')

In [ ]:
# Split features and target
X = df_encoded.drop('Attrition', axis=1)
y = df_encoded['Attrition']

# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]:,} rows  |  Test set: {X_test.shape[0]:,} rows')
print(f'Train attrition rate: {y_train.mean():.1%}')
print(f'Test attrition rate:  {y_test.mean():.1%}')

In [ ]:
# Apply SMOTE to handle class imbalance on training set only
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'After SMOTE — Training set: {X_train_res.shape[0]:,} rows')
print(f'Class balance: {pd.Series(y_train_res).value_counts().to_dict()}')

---
## 5. Model Training

In [ ]:
# Build pipeline: scale then logistic regression
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=42, C=0.5))
])

# Cross-validation on resampled training data
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, X_train_res, y_train_res,
                             cv=cv, scoring='roc_auc')

print(f'Cross-validation ROC-AUC scores: {cv_scores.round(3)}')
print(f'Mean: {cv_scores.mean():.3f}  |  Std: {cv_scores.std():.3f}')

# Fit on full resampled training set
pipeline.fit(X_train_res, y_train_res)
print('\nModel trained.')

---
## 6. Model Evaluation

In [ ]:
# Predictions
y_pred      = pipeline.predict(X_test)
y_pred_prob = pipeline.predict_proba(X_test)[:, 1]

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Stayed', 'Left']))
print(f'ROC-AUC Score: {roc_auc_score(y_test, y_pred_prob):.3f}')

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Stayed', 'Left'])
disp.plot(ax=ax, colorbar=False, cmap='YlOrBr')
ax.set_title('Confusion Matrix', color='#E6EDF3', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
auc_score   = roc_auc_score(y_test, y_pred_prob)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color=AMBER, linewidth=2, label=f'ROC AUC = {auc_score:.2f}')
ax.plot([0, 1], [0, 1], color=MUTED, linestyle='--', linewidth=0.8, label='Random classifier')
ax.set_title('ROC Curve', color='#E6EDF3', fontsize=13, pad=12)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Feature Importance — Top Attrition Drivers

In [ ]:
# Extract coefficients from the logistic regression model
model     = pipeline.named_steps['model']
scaler    = pipeline.named_steps['scaler']
feature_names = X.columns.tolist()

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False).head(15)

coef_df

In [ ]:
# Feature importance chart
top15 = coef_df.copy()
top15 = top15.sort_values('Coefficient')

fig, ax = plt.subplots(figsize=(9, 7))
colors = [AMBER if c > 0 else BLUE for c in top15['Coefficient']]
ax.barh(top15['Feature'], top15['Coefficient'],
        color=colors, edgecolor='none')
ax.axvline(0, color=MUTED, linewidth=0.8)
ax.set_title('Top 15 Attrition Drivers\n(Positive = increases attrition risk)',
             color='#E6EDF3', fontsize=13, pad=12)
ax.set_xlabel('Logistic Regression Coefficient')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=AMBER, label='Increases attrition risk'),
    Patch(facecolor=BLUE,  label='Decreases attrition risk')
]
ax.legend(handles=legend_elements, fontsize=9)
ax.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.show()

---
## 8. Flight Risk Scoring

Apply the model to the full dataset to produce an individual attrition probability score for every employee. This is the output that an HRBP would use.

In [ ]:
# Score all employees
attrition_probs = pipeline.predict_proba(X)[:, 1]

risk_df = df[['Department', 'JobRole', 'OverTime',
              'YearsInCurrentRole', 'MonthlyIncome',
              'JobSatisfaction']].copy()
risk_df['AttritionProbability'] = attrition_probs
risk_df['RiskBand'] = pd.cut(
    attrition_probs,
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Low', 'Medium', 'High']
)

print('Risk Band Distribution:')
print(risk_df['RiskBand'].value_counts())
print(f'\nHigh risk employees: {(risk_df["RiskBand"] == "High").sum():,}')

In [ ]:
# Risk band by department
dept_risk = risk_df.groupby(['Department', 'RiskBand']).size().unstack(fill_value=0)
dept_risk_pct = dept_risk.div(dept_risk.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8, 4))
dept_risk_pct.plot(kind='bar', ax=ax, stacked=True,
                   color=[TEAL, AMBER, RED],
                   edgecolor='none', width=0.5)
ax.set_title('Flight Risk Profile by Department', color='#E6EDF3', fontsize=13, pad=12)
ax.set_xlabel('')
ax.set_ylabel('% of Department')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Risk Band', fontsize=9, title_fontsize=9)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Export risk scores for Power BI
risk_df.to_csv('../data/attrition_risk_scores.csv', index=False)
print('Risk scores exported to /data/attrition_risk_scores.csv')
print('This file can be imported directly into Power BI for dashboard development.')

---
## 9. Key Findings and Business Recommendations

### What the model found

| Driver | Finding |
|---|---|
| Overtime | Employees on overtime are **2.3x more likely to leave**, regardless of pay level |
| Years in current role | Risk peaks at 1 year in role and again at 3+ years with no progression |
| Monthly income | Strong predictor below ~$3,500/month. Diminishing returns above that threshold |
| Job satisfaction | Scores of 1 or 2 out of 4 combined with overtime = highest risk profile |
| Job role | Sales Representatives (39%) and Lab Technicians (24%) had highest attrition rates |

### Business Recommendations

**1. Audit overtime by team, not by individual.**  
Overtime attrition is a structural problem, not an individual one. Identify which managers or departments run on sustained overtime and treat that as a workforce planning issue.

**2. Set a role tenure trigger at 3 years.**  
Flag any employee who has been in the same role for 3 or more years without movement for a proactive career conversation. A plan is more protective than a promotion.

**3. Segment pay reviews around the $3,500 threshold.**  
Compensation is a meaningful retention lever below $3,500/month. Above that level, invest retention effort in non-financial drivers: flexibility, recognition, and growth opportunity.

**4. Target Sales and Lab with specific interventions.**  
These roles carry the highest attrition risk and also the highest overtime exposure. A targeted workload review in these two functions will have disproportionate impact on overall attrition numbers.

---
*Oluwatobi (Tobi) Abanishe · HR Business Analyst · linkedin.com/in/oluwatobiabanishe*